# 02 · Train: baseline vs fine-tuned ResNet-18 (196 car models)

In [ ]:
import os, sys
REPO_URL = "https://github.com/YUPWE/car-models.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/car-models"
    if not os.path.exists(ROOT):
        !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
    !git pull -q
    !pip install -q imagehash==4.3.2
    os.environ["CARS_DATA_DIR"] = "/content/cars_data"
else:
    ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.makedirs("results", exist_ok=True)
os.makedirs("weights", exist_ok=True)
QUICK = os.environ.get("QUICK", "0") == "1"
print("Working in", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
import json, numpy as np, torch, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src import data
from src.data import CarDataset, get_transforms
from src.model import build_resnet18, train, set_seed, get_device
import torchvision, transformers, sklearn, pandas, matplotlib, PIL

data.download()
device = get_device()
print("device:", device)
for m in [torch, torchvision, transformers, sklearn, pandas, matplotlib, PIL]: print(f"  {m.__name__}=={m.__version__}")

train_df, val_df = data.load_split("train"), data.load_split("val")
if QUICK:
    train_df, val_df = train_df.sample(128, random_state=0), val_df.sample(64, random_state=0)
train_loader = DataLoader(CarDataset(train_df, get_transforms(True)), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(CarDataset(val_df, get_transforms(False)), batch_size=128, num_workers=2)
print(len(train_df), "train /", len(val_df), "val images")

In [ ]:
EPOCHS = 1 if QUICK else 10

set_seed(42)
baseline = build_resnet18(freeze_backbone=True, num_classes=196)
baseline, hist_base = train(baseline, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, device=device)
torch.save(baseline.state_dict(), "weights/resnet18_frozen.pt")

In [ ]:
set_seed(42)
finetuned = build_resnet18(freeze_backbone=False, num_classes=196)
finetuned, hist_ft = train(finetuned, train_loader, val_loader, epochs=EPOCHS, lr=5e-4, device=device)
torch.save(finetuned.state_dict(), "weights/resnet18_finetuned.pt")

In [ ]:
json.dump({"baseline_frozen": hist_base, "finetuned": hist_ft}, open("results/history.json", "w"), indent=1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, h, style in [("frozen (baseline)", hist_base, "--"), ("fine-tuned", hist_ft, "-")]:
    ep = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(ep, h["train_loss"], style, label=f"{name} train"); axes[0].plot(ep, h["val_loss"], style, label=f"{name} val")
    axes[1].plot(ep, h["train_acc"], style, label=f"{name} train"); axes[1].plot(ep, h["val_acc"], style, label=f"{name} val")
    axes[2].plot(ep, h["val_f1"], style, label=name)
for ax, t in zip(axes, ["Loss", "Accuracy", "Validation macro-F1"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/curves.png", dpi=120); plt.show()
for name, h in [("baseline (frozen)", hist_base), ("fine-tuned", hist_ft)]:
    print(f"{name:18s} best val macro-F1 {max(h['val_f1']):.3f} | best val acc {max(h['val_acc']):.3f}")